# Excercises 

## 0. Setup your own repo
- Dont work in this repo. This is my material for lessons. Set up your own repo to work in. Use `MADS-ML-{yourname}` as a format, eg `MADS-ML-JoostB`.
- You can add `mads_datasets` and `mltrainer` as dependencies to your own repo, in addition to more basic things like jupyter, torch and seaborn.
- If you want to use the tomlserializer, add `tomlserializer` as a dependency. For tensorboard, add `tensorboard` and `torch-tb-profiler`.
- Invite me (raoulg; https://github.com/raoulg) as a collaborator to your repo.

# 1. Tune the network
Run the experiment below, explore the different parameters (see suggestions below) and study the result with tensorboard. 
Make a single page (1 a4) report of your findings. Use your visualisation skills to communicate your most important findings.

In [5]:
from mads_datasets import DatasetFactoryProvider, DatasetType

from mltrainer.preprocessors import BasePreprocessor
from mltrainer import imagemodels, Trainer, TrainerSettings, ReportTypes, metrics

import torch.optim as optim
from torch import nn
from tomlserializer import TOMLSerializer

We will be using `tomlserializer` to easily keep track of our experiments, and to easily save the different things we did during our experiments.
It can export things like settings and models to a simple `toml` file, which can be easily shared, checked and modified.

First, we need the data. 

In [6]:
fashionfactory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
preprocessor = BasePreprocessor()
streamers = fashionfactory.create_datastreamer(batchsize=64, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()

2026-04-12 20:24:51.484 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 20:24:51.484 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt


We need a way to determine how well our model is performing. We will use accuracy as a metric.

In [7]:
accuracy = metrics.Accuracy()

You can set up a single experiment.

- We will show the model batches of 64 images, 
- and for every epoch we will show the model 100 batches (trainsteps=100).
- then, we will test how well the model is doing on unseen data (teststeps=100).
- we will report our results during training to tensorboard, and report all configuration to a toml file.
- we will log the results into a directory called "modellogs", but you could change this to whatever you want.

In [8]:
import torch
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=100,
    valid_steps=100,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


We will use a very basic model: a model with three linear layers.

In [9]:
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, units1),
            nn.ReLU(),
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Linear(units2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork(
    num_classes=10, units1=256, units2=256)

I developped the `tomlserializer` package, it is a useful tool to save configs, models and settings as a tomlfile; that way it is easy to track what you changed during your experiments.

This package will 1. check if there is a `__dict__` attribute available, and if so, it will use that to extract the parameters that do not start with an underscore, like this:

In [10]:
{k: v for k, v in model.__dict__.items() if not k.startswith("_")}

{'training': True, 'num_classes': 10, 'units1': 256, 'units2': 256}

This means that if you want to add more parameters to the `.toml` file, eg `units3`, you can add them to the class like this:

```python
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int, units3: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.units3 = units3  # <-- add this line
```

And then it will be added to the `.toml` file. Check the result for yourself by using the `.save()` method of the `TomlSerializer` class like this:

In [11]:
tomlserializer = TOMLSerializer()
tomlserializer.save(settings, "settings.toml")
tomlserializer.save(model, "model.toml")

Check the `settings.toml` and `model.toml` files to see what is in there.

You can use the `Trainer` class from my `mltrainer` module to train your model. It has the TOMLserializer integrated, so it will automatically save the settings and model to a toml file if you have added `TOML` as a reporttype in the settings.

In [12]:
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=trainstreamer,
    validdataloader=validstreamer,
    scheduler=optim.lr_scheduler.ReduceLROnPlateau
)
trainer.loop()

2026-04-12 20:24:51.569 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-202451
2026-04-12 20:24:52.598 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 100/100 [00:00<00:00, 124.58it/s]
2026-04-12 20:24:53.856 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.9584 test 0.6875 metric ['0.7359']
100%|██████████| 100/100 [00:00<00:00, 224.18it/s]
2026-04-12 20:24:54.813 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.5807 test 0.5441 metric ['0.8020']
100%|██████████| 100/100 [00:00<00:00, 110.20it/s]
2026-04-12 20:24:55.956 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.5189 test 0.5911 metric ['0.7981']
2026-04-12 20:24:55.957 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.5441, current loss 0.5911.Counter 1/10.
100%|██████████| 3/3 [00:03<00:00,  1.04s/it]


Now, check in the modellogs directory the results of your experiment.

We can now loop this with a naive approach, called a grid-search (why do you think i call it naive?).

In [13]:
"""  
The file with results is not supported?
"""

units = [256, 128, 64]
for unit1 in units:
    for unit2 in units:
        print(f"Units: {unit1}, {unit2}")

Units: 256, 256
Units: 256, 128
Units: 256, 64
Units: 128, 256
Units: 128, 128
Units: 128, 64
Units: 64, 256
Units: 64, 128
Units: 64, 64


Of course, this might not be the best way to search for a model; some configurations will be better than others (can you predict up front what will be the best configuration?).

So, feel free to improve upon the gridsearch by adding your own logic.

In [ ]:
import torch

units = [256, 128, 64]
loss_fn = torch.nn.CrossEntropyLoss()

settings = TrainerSettings(
    epochs=3,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)

for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()


2026-04-12 20:24:55.978 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-202455
2026-04-12 20:24:55.979 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 937/937 [00:04<00:00, 205.58it/s]
2026-04-12 20:25:00.895 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.5087 test 0.4065 metric ['0.8536']
100%|██████████| 937/937 [00:05<00:00, 168.67it/s]
2026-04-12 20:25:06.964 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.3651 test 0.3649 metric ['0.8692']
100%|██████████| 937/937 [00:04<00:00, 216.22it/s]
2026-04-12 20:25:11.660 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.3282 test 0.3823 metric ['0.8629']
2026-04-12 20:25:11.662 | INFO     | mltrainer.trainer:__call__:252 - best loss: 0.3649, current loss 0.3823.Counter 1/10.
100%|██████████| 3/3 [00:15<00:00,  5.23s/it]
2026-04-12 20:25:11.666 | INFO     | mltrainer.trainer:dir

Because we have set the ReportType to TOML, you will find in every log dir a model.toml and settings.toml file.

Run the experiment, and study the result with tensorboard. 

Locally, it is easy to do that with VS code itself. On the server, you have to take these steps:

- in the terminal, `cd` to the location of the repository
- activate the python environment for the shell. Note how the correct environment is being activated.
- run `tensorboard --logdir=modellogs` in the terminal
- tensorboard will launch at `localhost:6006` and vscode will notify you that the port is forwarded
- you can either press the `launch` button in VScode or open your local browser at `localhost:6006`


# Report
## 1. experiment
Experiment with things like:
- change the number of epochs, eg to 5 or 10. 
- changing the amount of units1 and units2 to values between 16 and 1024. Use factors of 2 to easily scan the ranges: 16, 32, 64, etc.
- changing the batchsize to values between 4 and 128. Again, use factors of two for convenience.
- change the depth of your model by adding a additional linear layer + activation function
- changing the learningrate to values between 1e-2 and 1e-5 
- changing the optimizer from SGD to one of the other available algoritms at [torch](https://pytorch.org/docs/stable/optim.html) (scroll down for the algorithms)

Check the results:
- all your experiments are saved in the `modellogs` directory, with a timestamp. Inside you find a model.toml file, that 
contains all the settings of the model. The `events` file is what tensorboard will show.
- visualize the relationship between variables: for example, make a heatmap of units vs layers.

Studyquestions:
- Epochs: what is the upside, what is the downside of increasing epochs? Do you need more epochs to find out which configuration is best? When do you need that, when not?
- what is an upside of using factors of 2 for hypertuning? What is a downside?

## Note
A note on train_steps: this is a setting that determines how often you get an update. 
Because our complete dataset is 938 (60000 / 64) batches long, you will need 938 trainstep to cover the complete 60.000 images.

This can actually be a bit confusion for some students, because changing the value of trainsteps 938 changes the meaning of an `epoch` slightly, because one epoch is no longer the full dataset, but simply `trainstep` batches. Setting trainsteps to 100 means you need to wait twice as long before you get feedback on the performance, as compared to trainsteps=50. You will see that settings trainsteps to 100 improves the learning, but that is simply because the model has seen twice as much examples as compared to trainsteps=50.

This implies that it is not usefull to compare trainsteps=50 and trainsteps=100, because setting it to 100 will always be better.
Just pick an amount that works for your hardware & patience, and adjust your number of epochs accordingly (increase epochs with lower values for trainsteps)

# 2. Reflect
Doing a master means you don't just start engineering a pipeline, but you need to reflect. Why do you see the results you see? What does this mean, considering the theory? Write down lessons learned and reflections, based on experimental results. This is the `science` part of `data science`.

You follow this cycle:
- make a hypothesis
- design an experiment
- run the experiment
- analyze the results and draw conclusions
- repeat

## Tip
To keep track of this process, it is useful to keep a journal. While you could use anything to do so, a nice command line tool is [jrnl](https://jrnl.sh/en/stable/). This gives you the advantage of staying in the terminal, just type down your ideas during the process, and you can always look back at what you have done.
Try to first formulate a hypothesis, and then design an experiment to test it. This will help you to stay focused on the goal, and not get lost in the data.

Important: the report you write is NOT the same as your journal! The journal will help you to keep track of your process, and later write down a reflection on what you have done where you draw conclusion, reflecting back on the theory.

# 3. Make a short report
Make a short 1 a4 page report of your findings.
pay attention to:
- what was your hypothesis about interaction between hyperparameters?
- what did you find?
- visualise your results about the relationship between hyperparameters.


In [ ]:
""" 
To experiment on:

To save time and unburden my laptop, I will use trainsteps 50

loss function will remain the same


ASPECT__________________________________| VALUES_USED___________| VALUES_TO_EXPERIMENT_WITH_(OUT_OF_INTERST)|
Epochs                                  | 3                     | 5, 10, 15                                 |
Unit values                             | [256, 128, 64]        | less: [64, 32, 16]                        |
.                                       |                       | more: [1024, 512, 256]                    |
Amount of units                         | 2                     | 4, 8                                      |
Batchsize                               | 64                    | 32, 128                                   |
Depth (layer + activation function)     | 3                     | 5, 10, 15                                 |
________________________________________|_______________________|___________________________________________|

"""

' \nTo experiment on:\n\nEpochs\nUnit values\nAmount of units\nBatchsize\nLayers (layer + activation function)\n\n- change the number of epochs, eg to 5 or 10. \n- changing the amount of units1 and units2 to values between 16 and 1024. Use factors of 2 to easily scan the ranges: 16, 32, 64, etc.\n- changing the batchsize to values between 4 and 128. Again, use factors of two for convenience.\n- change the depth of your model by adding a additional linear layer + activation function\n- changing the learningrate to values between 1e-2 and 1e-5 \n- changing the optimizer from SGD to one of the other available algoritms at [torch](https://pytorch.org/docs/stable/optim.html) (scroll down for the algorithms)\n\n\n'

In [ ]:
"""EXPERIMENT 1: THE OG """
epochs        = 3
units         = [256, 128, 64]
batchsize     = 64
optimizer     = optim.Adam


# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:40:28.527 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:40:28.528 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 21:40:28.606 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-214028
2026-04-12 21:40:28.607 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:01<00:00, 43.21it/s]
2026-04-12 21:40:30.070 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1510 test 0.7603 metric ['0.7331']
100%|██████████| 50/50 [00:00<00:00, 218.31it/s]
2026-04-12 21:40:30.422 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6598 test 0.6223 metric ['0.7684']
100%|██████████| 50/50 [00:00<00:00, 214.25it/s]
2026-04-12 21:40:30.777 | INFO     | mltr

In [39]:
"""EXPERIMENT 2: EPOCHS (5) """
epochs        = 5
units         = [256, 128, 64]
batchsize     = 64
optimizer     = optim.Adam


# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 22:51:06.116 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 22:51:06.118 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 22:51:06.160 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-225106
2026-04-12 22:51:06.161 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:00<00:00, 221.95it/s]
2026-04-12 22:51:06.510 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1684 test 0.8561 metric ['0.6737']
100%|██████████| 50/50 [00:00<00:00, 222.28it/s]
2026-04-12 22:51:06.855 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6696 test 0.6592 metric ['0.7553']
100%|██████████| 50/50 [00:00<00:00, 222.70it/s]
2026-04-12 22:51:07.198 | INFO     | mlt

In [26]:
"""EXPERIMENT 3: EPOCHS (10) """
epochs        = 10
units         = [256, 128, 64]
batchsize     = 64
optimizer     = optim.Adam


# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:48:49.898 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:48:49.899 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 21:48:49.981 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-214849
2026-04-12 21:48:49.982 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:00<00:00, 86.52it/s]
2026-04-12 21:48:50.824 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1704 test 0.7877 metric ['0.6997']
100%|██████████| 50/50 [00:00<00:00, 223.81it/s]
2026-04-12 21:48:51.168 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6806 test 0.6196 metric ['0.7712']
100%|██████████| 50/50 [00:00<00:00, 222.86it/s]
2026-04-12 21:48:51.509 | INFO     | mltr

In [27]:
"""EXPERIMENT 4: EPOCHS (15) """
epochs        = 15
units         = [256, 128, 64]
batchsize     = 64
optimizer     = optim.Adam

# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:49:49.706 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:49:49.706 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 21:49:49.744 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-214949
2026-04-12 21:49:49.745 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:00<00:00, 217.23it/s]
2026-04-12 21:49:50.099 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.1970 test 0.7713 metric ['0.7081']
100%|██████████| 50/50 [00:00<00:00, 228.47it/s]
2026-04-12 21:49:50.438 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6683 test 0.6088 metric ['0.7678']
100%|██████████| 50/50 [00:00<00:00, 226.13it/s]
2026-04-12 21:49:50.776 | INFO     | mlt

In [30]:
"""EXPERIMENT 5: UNIT VALUES (less: [64, 32, 16])"""
epochs        = 3
units         = [64, 32, 16]
batchsize     = 64
optimizer     = optim.Adam

# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:53:39.511 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:53:39.514 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt


2026-04-12 21:53:39.579 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-215339
2026-04-12 21:53:39.580 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:00<00:00, 134.21it/s]
2026-04-12 21:53:40.257 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.5103 test 0.9421 metric ['0.6512']
100%|██████████| 50/50 [00:00<00:00, 128.96it/s]
2026-04-12 21:53:41.345 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.8016 test 0.7331 metric ['0.7441']
100%|██████████| 50/50 [00:00<00:00, 138.54it/s]
2026-04-12 21:53:41.886 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 0.6926 test 0.6899 metric ['0.7534']
100%|██████████| 3/3 [00:02<00:00,  1.31it/s]
2026-04-12 21:53:41.893 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-215341
2026-04-12 21:53:41.895 | INFO     | mltrainer.trainer:__init__:68 - Fou

In [31]:
"""EXPERIMENT 6: UNIT VALUES (more: [1024, 512, 256])"""
epochs        = 3
units         = [1024, 512, 256]
batchsize     = 64
optimizer     = optim.Adam

# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:54:02.493 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:54:02.494 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 21:54:02.592 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-215402
2026-04-12 21:54:02.594 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:01<00:00, 38.97it/s]
2026-04-12 21:54:04.102 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 0.9539 test 0.6421 metric ['0.7478']
100%|██████████| 50/50 [00:00<00:00, 68.06it/s]
2026-04-12 21:54:05.063 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6055 test 0.6382 metric ['0.7625']
100%|██████████| 50/50 [00:01<00:00, 40.22it/s]
2026-04-12 21:54:06.667 | INFO     | mltrai

In [ ]:
"""EXPERIMENT 7: BATCHSIZE (32)"""
epochs        = 3
units         = [256, 128, 64]
batchsize     = 32
optimizer     = optim.Adam

# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:55:31.874 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:55:31.875 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 21:55:31.916 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-215531
2026-04-12 21:55:31.917 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:00<00:00, 297.01it/s]
2026-04-12 21:55:32.157 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.3398 test 0.8692 metric ['0.6650']
100%|██████████| 50/50 [00:00<00:00, 307.97it/s]
2026-04-12 21:55:32.387 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.7482 test 0.7115 metric ['0.7356']
100%|██████████| 50/50 [00:00<00:00, 330.99it/s]
2026-04-12 21:55:32.608 | INFO     | mlt

In [ ]:
""""
ASPECT__________________________________| VALUES_USED___________| VALUES_TO_EXPERIMENT_WITH_(OUT_OF_INTERST)|
Epochs                                  | 3                     | 5, 10, 15                                 |
Unit values                             | [256, 128, 64]        | less: [64, 32, 16]                        |
.                                       |                       | more: [1024, 512, 256]                    |
Batchsize                               | 64                    | 32, 128                                   |
Optimizer                               | Adam                  | SGD                                       |  
Amount of units (correlates with Depth) | 2                     | 4, 6, 8                                   |
Depth (layer + activation function)     | 3                     | 5, 7, 9                                   |
________________________________________|_______________________|___________________________________________|


Hypotheses:
- I expect accuracy to be higher when the number of Epochs are higher
- I expect accuracy to be higher when the unit values are higher
- I expect accuracy to be higher when the Depth is higher
"""
"""EXPERIMENT 7: BATCHSIZE (128)"""
epochs        = 3
units         = [256, 128, 64]
batchsize     = 128
optimizer     = optim.Adam

# for changing the batchsize
streamers = fashionfactory.create_datastreamer(batchsize=batchsize, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]
trainstreamer = train.stream()
validstreamer = valid.stream()


# for changing epochs, train_steps and valid_steps
settings = TrainerSettings(
    epochs=epochs,
    metrics=[accuracy],
    logdir="modellogs",
    train_steps=50,
    valid_steps=50,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)


# training the model - looping
for unit1 in units:
    for unit2 in units:

        model = NeuralNetwork(num_classes=10, units1=unit1, units2=unit2)

        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=trainstreamer,
            validdataloader=validstreamer,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau
        )
        trainer.loop()

2026-04-12 21:56:02.393 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/gjongde/.cache/mads_datasets/fashionmnist
2026-04-12 21:56:02.394 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/gjongde/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-04-12 21:56:02.432 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to modellogs/20260412-215602
2026-04-12 21:56:02.433 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 50/50 [00:00<00:00, 137.57it/s]
2026-04-12 21:56:03.023 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.0855 test 0.6743 metric ['0.7572']
100%|██████████| 50/50 [00:00<00:00, 143.98it/s]
2026-04-12 21:56:03.595 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 0.6060 test 0.6146 metric ['0.7739']
100%|██████████| 50/50 [00:00<00:00, 131.49it/s]
2026-04-12 21:56:04.193 | INFO     | mlt

In [19]:
# units order
units = units
for unit1 in units:
    for unit2 in units:
        print(f"Units: {unit1}, {unit2}")

Units: 256, 256
Units: 256, 128
Units: 256, 64
Units: 128, 256
Units: 128, 128
Units: 128, 64
Units: 64, 256
Units: 64, 128
Units: 64, 64


In [20]:
# model for changing amount of units and depth
class NeuralNetwork(nn.Module):
    def __init__(self, num_classes: int, units1: int, units2: int) -> None:
        super().__init__()
        self.num_classes = num_classes
        self.units1 = units1
        self.units2 = units2
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, units1),
            nn.ReLU(),
            nn.Linear(units1, units2),
            nn.ReLU(),
            nn.Linear(units2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork(num_classes=10, units1=256, units2=256)

In [ ]:
import numpy as np
import seaborn as sns

oefen = np.array([[1, 2, 3],
         [32, 64, 128],
         [0,1,1]])

print(oefen)

sns.heatmap(oefen)